In [5]:
from cocohirf.experiment.hpo_open_ml_vecohirf_experiment import HPOOpenmlVeCoHiRFExperiment
from cocohirf.experiment.hpo_open_ml_coclustering_experiment import HPOOpenmlCoClusteringExperiment
import pandas as pd
from IPython.display import clear_output
from cocohirf.experiment.tested_models import models_dict, two_stage_models_dict
from ml_experiments.utils import print_keys

In [6]:
def run_experiment(models, seeds, hpo_metric, direction, evaluation_metric, n_agents, dataset_id):
    results = []
    agent_is = [None] if n_agents is None else [i for i in range(n_agents)]
    for model in models:
        if model in ["KMeans", "DBSCAN", "SpectralSubspaceRandomization"]:
            agent_is = [i for i in range(n_agents)]
        else:
            agent_is = [None]
        for seed in seeds:
            for agent_i in agent_is:
                print(f"Running experiment with model={model}, seed={seed}, agent_i={agent_i}")
                try:
                    if model.find("VeCoHiRF") != -1:
                        experiment = HPOOpenmlVeCoHiRFExperiment(
                            model_alias=model,
                            dataset_id=dataset_id,
                            seed_dataset_order=seed,
                            n_agents=n_agents,
                            agent_i=agent_i,
                            standardize=True,
                            hpo_metric_1=hpo_metric[: -len("_mean")],
                            hpo_metric_2=hpo_metric,
                            direction_1=direction,
                            direction_2=direction,
                            calculate_full_silhouette=True,
                            hpo_seed=seed,
                            n_trials_1=50,
                            n_trials_2=30,
                            sampler_1="tpe",
                            sampler_2="tpe",
                            pruner_1="none",
                            pruner_2="none",
                            verbose=1,
                            raise_on_error=True,
                            n_jobs=5,
                        )
                    else:
                        experiment = HPOOpenmlCoClusteringExperiment(
                            model=model,
                            dataset_id=dataset_id,
                            seed_dataset_order=seed,
                            n_agents=n_agents,
                            agent_i=agent_i,
                            standardize=True,
                            hpo_metric=hpo_metric,
                            direction=direction,
                            calculate_full_silhouette=True,
                            hpo_seed=seed,
                            n_trials=50,
                            verbose=1,
                            raise_on_error=True,
                            n_jobs=5,
                        )
                    result = experiment.run(return_results=True)[0]
                    hpo_score = result["evaluate_model_return"][f"best/{hpo_metric}"]
                    evaluation_score = result["evaluate_model_return"][f"best/{evaluation_metric}"]
                    hpo_time = result["fit_model_return"]["elapsed_time"]
                    best_time = result["evaluate_model_return"]["best/elapsed_time"]
                    results.append(
                        dict(
                            hpo_score=hpo_score,
                            evaluation_score=evaluation_score,
                            hpo_time=hpo_time,
                            best_time=best_time,
                            seed=seed,
                            model=model,
                            agent_i=agent_i,
                        )
                    )
                except Exception as e:
                    print(f"Error running experiment with model={model}, seed={seed}, agent_i={agent_i}: {e}")
                    results.append(
                        dict(
                            hpo_score=None,
                            evaluation_score=None,
                            hpo_time=None,
                            best_time=None,
                            seed=seed,
                            model=model,
                            agent_i=agent_i,
                        )
                    )
                finally:
                    clear_output(wait=True)
    return pd.DataFrame(results)

# Optimizing Silhouette and evaluating ARI on Iris

In [7]:
seeds = [i for i in range(3)]
hpo_metric = "silhouette_mean"
direction = "maximize"
evaluation_metric = "adjusted_rand_mean"
n_agents = 2
models = [
    "KMeans",
    "DBSCAN",
    "SpectralSubspaceRandomization",
    "DistributedKMeans",
    "CoresetKMeans",
    "VeCoHiRF",
    "VeCoHiRF-1iter",
    "VeCoHiRF-DBSCAN",
    "VeCoHiRF-DBSCAN-1iter",
    "VeCoHiRF-KernelRBF",
    "VeCoHiRF-KernelRBF-1iter",
    "VeCoHiRF-SC-SRGF-1R",
    "VeCoHiRF-SC-SRGF-1R-1iter",
    "VeCoHiRF-SC-SRGF-2R",
    "VeCoHiRF-SC-SRGF-2R-1iter",
]

In [ ]:
results = run_experiment(models, seeds, hpo_metric, direction, evaluation_metric, n_agents, dataset_id=46779)

Running experiment with model=KMeans, seed=1, agent_i=0


Combinations completed:   0%|          | 0/1 [00:00<?, ?it/s]

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x767587adc790>>
Traceback (most recent call last):
  File "/home/belucci/miniconda3/envs/cocohirf/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


Trials:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
results.groupby("model").agg(["mean", "std"])

hpo_score               evaluation_score            \
                              mean           std             mean       std   
model                                                                         
CoresetKMeans             0.577602  1.391880e-03         0.552342  0.007132   
DistributedKMeans         0.579209  0.000000e+00         0.560577  0.000000   
KMeans                    0.571009  8.982969e-03         0.543358  0.018862   
VeCoHiRF                  0.534617  1.359740e-16         0.498361  0.000000   
VeCoHiRF-1iter            0.327364  1.835518e-01         0.386620  0.212537   
VeCoHiRF-KernelRBF        0.388405  1.379119e-01         0.430516  0.125461   
VeCoHiRF-KernelRBF-1iter  0.185578  1.172243e-01         0.236365  0.133356   

                           hpo_time           best_time           seed  \
                               mean       std      mean       std mean   
model                                                                    
CoresetKMeans              2.770663  0.123543  0.013094  0.000212  1.0   
DistributedKMeans          3.148132  1.787322  0.013186  0.000234  1.0   
KMeans                     1.647441  0.222486  0.002625  0.000516  1.0   
VeCoHiRF                  14.900584  4.770215  0.110882  0.005767  1.0   
VeCoHiRF-1iter             9.272761  2.701011  0.132212  0.094015  1.0   
VeCoHiRF-KernelRBF        24.864073  3.515393  0.226926  0.196590  1.0   
VeCoHiRF-KernelRBF-1iter  15.464482  3.669329  0.328320  0.226755  1.0   

                                   agent_i            
                               std    mean       std  
model                                                 
CoresetKMeans             1.000000     NaN       NaN  
DistributedKMeans         1.000000     NaN       NaN  
KMeans                    0.894427     0.5  0.547723  
VeCoHiRF                  1.000000     NaN       NaN  
VeCoHiRF-1iter            1.000000     NaN       NaN  
VeCoHiRF-KernelRBF        1.000000     NaN       NaN  
VeCoHiRF-KernelRBF-1iter  1.000000     NaN       NaN

# Optimizing ARI and evaluating Silhouette on Iris

In [ ]:
seeds = [i for i in range(3)]
hpo_metric = "adjusted_rand_mean"
direction = "maximize"
evaluation_metric = "silhouette_mean"
n_agents = 2
models = [
    "KMeans",
    "DistributedKMeans",
    "CoresetKMeans",
    "VeCoHiRF",
    "VeCoHiRF-1iter",
    "VeCoHiRF-KernelRBF",
    "VeCoHiRF-KernelRBF-1iter",
]

In [ ]:
results = run_experiment(models, seeds, hpo_metric, direction, evaluation_metric, n_agents, dataset_id=61)

Running experiment with model=VeCoHiRF-KernelRBF-1iter, seed=2, agent_i=None


Combinations completed:   0%|          | 0/1 [00:00<?, ?it/s]

Trials:   0%|          | 0/50 [00:00<?, ?it/s]

/home/belucci/code/cohirf/cohirf/experiment/clustering_experiment.py:27: UserWarning: Too many clusters (95) for dataset with 147 samples. Skipping metric calculation. If you want to calculate metrics anyway, set `calculate_metrics_even_if_too_many_clusters` to True.
  warn(f"Too many clusters ({n_clusters}) for dataset with {X.shape[0]} samples. Skipping metric calculation. If you want to calculate metrics anyway, set `calculate_metrics_even_if_too_many_clusters` to True.")
/home/belucci/code/ml_experiments/ml_experiments/tuners.py:233: UserWarning: metric adjusted_rand not found in dict returned by training_fn, available metrics are dict_keys(['n_clusters_', 'elapsed_time', 'seed_model'])
  warn(f'metric {metric} not found in dict returned by training_fn, available metrics are '
/home/belucci/code/cohirf/cohirf/experiment/clustering_experiment.py:27: UserWarning: Too many clusters (89) for dataset with 147 samples. Skipping metric calculation. If you want to calculate metrics anyway,

Trials:   0%|          | 0/50 [00:00<?, ?it/s]

Trials:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
results.groupby("model").agg(["mean", "std"])

hpo_score           evaluation_score            \
                              mean       std             mean       std   
model                                                                     
CoresetKMeans             0.645138  0.084056         0.466906  0.095340   
DistributedKMeans         0.564586  0.005259         0.383147  0.169798   
KMeans                    0.651415  0.146677         0.438977  0.106362   
VeCoHiRF                  0.813186  0.007138         0.260197  0.040218   
VeCoHiRF-1iter            0.726930  0.046403         0.237811  0.057598   
VeCoHiRF-KernelRBF        0.785293  0.062794         0.284178  0.044114   
VeCoHiRF-KernelRBF-1iter  0.761684  0.087854         0.290616  0.061510   

                           hpo_time           best_time           seed  \
                               mean       std      mean       std mean   
model                                                                    
CoresetKMeans              3.428072  0.137675  0.013321  0.000374  1.0   
DistributedKMeans          4.741199  3.496203  0.013685  0.000889  1.0   
KMeans                     2.160192  0.249459  0.003092  0.000661  1.0   
VeCoHiRF                  16.598339  6.481922  0.134691  0.021915  1.0   
VeCoHiRF-1iter             8.122296  0.191343  0.081573  0.012829  1.0   
VeCoHiRF-KernelRBF        21.847584  3.115155  0.221048  0.073082  1.0   
VeCoHiRF-KernelRBF-1iter   9.510991  0.879666  0.145580  0.071348  1.0   

                                   agent_i            
                               std    mean       std  
model                                                 
CoresetKMeans             1.000000     NaN       NaN  
DistributedKMeans         1.000000     NaN       NaN  
KMeans                    0.894427     0.5  0.547723  
VeCoHiRF                  1.000000     NaN       NaN  
VeCoHiRF-1iter            1.000000     NaN       NaN  
VeCoHiRF-KernelRBF        1.000000     NaN       NaN  
VeCoHiRF-KernelRBF-1iter  1.000000     NaN       NaN